In [1]:
import math 
import time
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import os
import torch
import cv2
import torchvision.transforms as transforms
from torch.utils.data import Dataset , DataLoader, random_split
from dotenv import load_dotenv
import numpy as np
import torch.nn as nn
import torch.optim as optim
from imgDataset import imgDataset
from EAST import EAST

imported imgDataset.py
torch version: 2.9.1+cu128


In [2]:
#loads in paths
load_dotenv()
directory_train = os.getenv('Directory_train')
directory_train_textAndCoords = os.getenv('directory_train_textAndCoords')
directory_test = os.getenv('Directory_test')
directory_test_textAndCoords = os.getenv('directory_test_textAndCoords')
width = 640
height = 360

In [3]:
# Define path_imgs and path_labels
path_imgs = os.listdir(directory_train)
path_labels = os.listdir(directory_train_textAndCoords)

# Generate paths for training images and labels
train_img_paths = [os.path.join(directory_train, f) for f in path_imgs]
train_label_paths = [os.path.join(directory_train_textAndCoords, f) for f in path_labels]

In [4]:
#making objs
train_dataset = [train_img_paths, train_label_paths]
train_dataset = imgDataset(train_img_paths, train_label_paths, img_width= width, img_height= height)
img, labels, coords, corners = train_dataset[0]

print(f"Image shape: {img.shape}", type(img)) # Should be [1, 360, 360] since it's grayscale
print(f"First Coords: {coords.shape}",type(coords))

#batch = custom_collate(train_dataset)


Image shape: torch.Size([1, 360, 640]) <class 'torch.Tensor'>
First Coords: torch.Size([1, 360, 640]) <class 'torch.Tensor'>


In [6]:
img, labels, coords, corners = train_dataset[0]

# img [1, H, W] → BGR
img_np = img.squeeze().numpy()
img_np = (img_np * 255).astype(np.uint8)
img_bgr = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)

coords_np = coords.squeeze().numpy()  # binary mask [H, W]

overlay = img_bgr.copy()

# white where text box is
overlay[coords_np > 0] = [255, 255, 255]

# for each pixel inside a box, draw lines from pixel → each of the 4 corners
# corners[k*2] = ∆x to corner k, corners[k*2+1] = ∆y to corner k
ys, xs = np.where(coords_np > 0)

corner_colors = [
    (0, 0, 255),    # corner 0 - red
    (0, 255, 0),    # corner 1 - green
    (255, 0, 0),    # corner 2 - blue
    (0, 255, 255),  # corner 3 - yellow
]

# subsample pixels so lines are readable
step = 10
for y, x in zip(ys[::step], xs[::step]):
    for k in range(4):
        dx = corners[k*2][y][x].item()
        dy = corners[k*2+1][y][x].item()
        # corner position = pixel + offset
        cx = int(x + dx)
        cy = int(y + dy)
        cv2.line(overlay, (x, y), (cx, cy), corner_colors[k], 1)

# draw the actual corner dots on top
for k in range(4):
    dx_map = corners[k*2].numpy()
    dy_map = corners[k*2+1].numpy()
    # corners exist where TrueMap is active — pick any pixel and follow its offset
    sample_ys, sample_xs = np.where(coords_np > 0)
    if len(sample_ys) > 0:
        sy, sx = sample_ys[0], sample_xs[0]
        cx = int(sx + corners[k*2][sy][sx].item())
        cy = int(sy + corners[k*2+1][sy][sx].item())
        cv2.circle(overlay, (cx, cy), radius=5, color=corner_colors[k], thickness=-1)

cv2.imshow("overlay", overlay)
cv2.waitKey(0)
cv2.destroyAllWindows()